In [ ]:
# Jupyter Notebook for eQTL study

In [ ]:
# Import custom utility packages, lists and functions
from init_env import *
from anndata_utils import *
from gene_lists import *

# Adjust Scanpy figure defaults
sc.settings.set_figure_params(dpi=100, fontsize=10, dpi_save=400,
    facecolor = 'white', figsize=(12,6), format='png')

In [ ]:
# Load data and concat data

In [ ]:
# Define memory-monitored function
@profile
def load_and_process_data(downsample_cells=None, *adata_dirs):
    """
    Load, process, and merge AnnData objects with optional downsampling.
    
    Parameters:
        downsample_cells (int, optional): Number of cells to randomly downsample to. If None, no downsampling is performed.
        *adata_dirs (str): Paths to AnnData directories. Each directory should contain an 'anndata.h5ad' file.
        
    Returns:
        AnnData: A merged and processed AnnData object.
    """
    adata_list = []   
    
    # Extract plate numbers from the directory paths
    plate_numbers = [re.search(r'plate(\d+)', adata_dir).group(1) for adata_dir in adata_dirs]

    # Report how many plates will be processed and downsampling details
    if downsample_cells != None:
        print(f"Processing {len(adata_dirs)} plates with downsampling to {downsample_cells} cells per plate.")
    else:
        print(f"Processing {len(adata_dirs)} plates, no downsampling applied.")

    for i, adata_dir in enumerate(adata_dirs):
        # Load the AnnData object
        print(f"Loading plate {len(plate_numbers[i)} ...")
        adata = sc.read(adata_dir + 'anndata.h5ad')
        
        # Assign unique plate information to cell ids and metadata
        adata.obs['plate'] = 'plate' + plate_numbers[i]
        adata.obs_names = f"plate{plate_numbers[i]}_" + adata.obs_names

        # Log dimensions
        print(f"Plate {plate_numbers[i]} dimensions: {adata.shape}")
        print(f"Plate {plate_numbers[i]} matrix Dimensions: {adata.X.shape}")
 
        # Optional downsampling
        if downsample_cells is not None and downsample_cells < adata.n_obs:
            random_indices = np.random.choice(adata.n_obs, downsample_cells, replace=False)
            adata = adata[random_indices, :].copy()
        
        adata_list.append(adata)

    # Find common genes across all datasets
    # Optionally handle common genes only if more than one plate
    if len(adata_list) > 1:
        common_genes = adata_list[0].var_names
        for adata in adata_list[1:]:
            common_genes = common_genes.intersection(adata.var_names)
        
        print(f"Common genes across all plates: {common_genes.shape[0]}")
        
        # Subset all adata objects for common genes
        for i, adata in enumerate(adata_list):
            adata_list[i] = adata[:, common_genes]
    else:
        print(f"Only one plate processed; no need to find common genes.")
        
    # Concatenate the AnnData objects
    print(f"Merging plates ...")
    adata_mrg = sc.concat(adata_list, join='inner')

    # Remove 'hg38' from gene names
    adata_mrg.var.index = adata_mrg.var.index.str.replace('_hg38', '')

    # Cleanup to release memory
    del adata_list
    gc.collect()

    print(adata_mrg)

    return adata_mrg


adata = load_and_process_data(plate1_dir, plate2_dir)

In [ ]:
adata

In [ ]:
# QC metadata

In [ ]:
adata.obs['sample'] = adata.obs['sample'].str.replace('sample_', '')
adata.obs['plate'] = adata.obs.index.str.split('_').str[0]
adata = adata[~adata.obs['sample'].str.endswith(tuple(['WGE', 'Hipp', 'Thal']))]
adata.obs['sample'].value_counts()

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True
)

plt.hist(adata.var['n_cells_by_counts'], bins=500)
plt.xlabel('N cells expressing > 0')
plt.ylabel('log(N genes)') # for visual clarity
plt.axvline(2, color='red')
plt.yscale('log') 

sns.jointplot(
   data=adata.obs,
   x="log1p_total_counts",
   y="log1p_n_genes_by_counts",
   kind="hex",
)

In [ ]:
# Normalise

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
# ID highly variable genes

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.25)
# This saves the original set of genes 
adata.raw = adata

adata = adata[:,adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)

In [ ]:
# PCA

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True, n_pcs=50, save='') # scanpy generates the filename automatically

In [ ]:
# UMAP and Clustering

In [ ]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=50)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.5)

In [ ]:
# Plotting

In [ ]:
# Helper function
sc.pl.umap(adata, color=['leiden'])

In [ ]:
# Violin plots
def plot_filtered_violin(adata, gene_set, groupby, row_palette=None, swap_axes=True, **kwargs):
    """
    Plot a stacked violin plot for a filtered gene set, ignoring missing genes and reporting missing ones.

    Parameters:
        adata (AnnData): The AnnData object containing the data.
        gene_set (list): List of genes to plot.
        groupby (str): Key in `adata.obs` to group by.
        row_palette (list or dict, optional): Color palette for rows. Default is None.
        swap_axes (bool, optional): Whether to swap axes. Default is True.
        **kwargs: Additional keyword arguments passed to `sc.pl.stacked_violin`.

    Returns:
        None: Displays the plot or a message if no valid genes are found.
    """
    # Separate valid and missing genes
    valid_genes = [gene for gene in gene_set if gene in adata.var_names or gene in getattr(adata.raw, 'var_names', [])]
    missing_genes = [gene for gene in gene_set if gene not in valid_genes]

    # Report missing genes
    if missing_genes:
        print(f"Genes not found in dataset): {', '.join(missing_genes)}")
    
    if valid_genes:
        # Plot the filtered gene set
        print(f"Plotting {len(valid_genes)} genes out of {len(gene_set)} provided.")
        sc.pl.stacked_violin(
            adata, 
            valid_genes, 
            groupby=groupby, 
            row_palette=row_palette, 
            swap_axes=swap_axes, 
            **kwargs
        )
    else:
        print("None of the genes in the provided gene set are present in the dataset.")


plot_filtered_violin(adata, general_genes, "leiden", row_palette=discreet_cols_n23)